# Test One: Simulation Algorithm Scaling with Number of Compartments (No Intercompartmental Interaction).

The first test aims to see how well the different simulation algorithms scale with compartment number.

## Test Specification

### Models

SEIR?

SIS?

Seasonal SEIR?

### Simulation Algorithms

Gillespie (Direct, no caching)

Gillespie (Direct, caching)

Gillespie(First, caching)

Gillespie(Next, caching required)

### Compartments Numbers (Each with 1000 initial population, of which an inital 10 are infected)

1, 2, 3, 5, 7, 10


In [5]:
import pyRBM.Core.Model

from pyRBM.Build.Compartment import Compartment, Location
import pyRBM.Build.RuleTemplates as Template


In [6]:
%pip install indexed_priority_queue

Note: you may need to restart the kernel to use updated packages.


In [7]:
epiClasses = [["S", "people"], ["E", "people"], ["I", "people"], ["R", "people"]]

In [8]:
class EpiComp(Compartment):
    def __init__(self, name:str, initial_infected = 10, total_population = 1000, constants = None):
        # Provides a list of constants that should be set (optional, but will provide a useful error
        # message if unset).
        if constants is None:
            constants = ["contact_rate", "infectivity_rate", "recovery_rate", "mortality_rate"]
        super().__init__(name, comp_type="EpiComp", constants=constants)
        # Crops exist in three stages in this simplified model: planted, growing and harvested.
        class_labels = [class_entry[0] for class_entry in epiClasses]
        self.addClassLabels(class_labels)

        self.setInitialConditions({"S":total_population-initial_infected,
                                   "I":initial_infected})

In [9]:
def seirRules(args):
    exposure = Template.SingleLocationProductionRule("EpiComp",
                                                        "S", 1, # Reactants
                                                        "E", 1, # Products
                                                       "S*I*comp_contact_rate", ["S", "I"], # Propensity
                                                       "Exposure of Susceptible")
    
    infection = Template.SingleLocationProductionRule("EpiComp",
                                                       "E", 1,
                                                       "I", 1,
                                                       "I*comp_infectivity_rate", "I", 
                                                       "Infection of Exposed")
     
    recovery = Template.SingleLocationProductionRule("EpiComp",
                                                       "I", 1,
                                                       "R", 1,
                                                       "I*comp_recovery_rate", "I", 
                                                       "Recovery of Infected")
    death = Template.ExitEntranceRule("EpiComp",
                                        "I", -1, # Reactants if negative, products if positive
                                        "I*comp_mortality_rate", "I",
                                        "Death of Infected")
    
    return  (exposure, infection, recovery, death)

In [10]:
def sisRules(args):
    infection = Template.SingleLocationProductionRule("EpiComp",
                                                       "S", 1,
                                                       "I", 1,
                                                       "S*I*comp_infectivity_rate", ["I","S"], 
                                                       "Infection of Exposed")
     
    recovery = Template.SingleLocationProductionRule("EpiComp",
                                                       "I", 1,
                                                       "R", 1,
                                                       "I*comp_recovery_rate", "I", 
                                                       "Recovery of Infected")
    
    return  (infection, recovery)

In [ ]:
def seasonalSeirRules(args):
    

In [11]:
import datetime
from pyRBM.Simulation import Solvers
from pyRBM.Core.Model import Model

In [12]:
def create_n_compartment_model(n):
    
    def create_compartments(args, n):
        default_constants = {
            "contact_rate" : 0.5,
            "infectivity_rate" : 0.4,
            "recovery_rate" : 0.1,
            "mortality_rate" : 0.3
            }
        return [EpiComp(f"Compartment {i}", constants = default_constants) for i in range(n)]

    return lambda args: create_compartments(args, n)

In [20]:
%pip install pandas

   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   --------- ------------------------------ 2.9/11.6 MB 16.8 MB/s eta 0:00:01
   -------------------- ------------------- 6.0/11.6 MB 16.1 MB/s eta 0:00:01
   --------------------------------- ------ 9.7/11.6 MB 15.9 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 15.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
import pandas as pd

In [14]:
# SEIR results
num_test_runs = 1


def runPerfTests(model, rules_func, n_compartments_func_creator, model_classes):
    solver_in_test = {"Direct - No Caching" : Solvers.GillespieSolver(use_cached_propensities=False, debug=False, no_rules_behaviour="end"), 
                  "Direct - Caching" : Solvers.GillespieSolver(use_cached_propensities=True, debug=False, no_rules_behaviour="end"),
                  "First Reaction" : Solvers.GillespieFRMSolver(debug=False, no_rules_behaviour="end"),
                  "Next Reaction" : Solvers.GillespieNRMSolver(debug=False, no_rules_behaviour="end")}
    
    results_dict = {"model":[], "simulation_algorithm":[], "num_compartments":[],
                    "avg_simulation_time":[], "std_simulation_time":[],
                    "avg_simulation_iterations":[], "std_simulation_iterations":[]}
    for compartment_num in (1, 2, 3, 5, 7, 10):
        model.buildModel(classes_defintions=epiClasses, create_rules=rules_func, create_compartments=n_compartments_func_creator(compartment_num))
        model.convertToSimulation()

        for solver_name, solver in solver_in_test.items():
            model.initializeSolver(solver)
            for test_iterations in range(50):
                model.simulate(start_date=datetime.date(2001,1,1), time_limit=100, max_iterations=15000)

            # return stats over all 50 simulations for the given simulation/model configuration
            perf_results = model.returnSimulationPerformanceStats()
            results_dict["model"].append(model.model_name)
            results_dict["simulation_algorithm"].append(solver_name)
            results_dict["num_compartments"].append(compartment_num)
            results_dict["avg_simulation_time"].append(perf_results["mean_simulation_time"])
            results_dict["std_simulation_time"].append(perf_results["std_simulation_time"])
            results_dict["avg_simulation_iterations"].append(perf_results["mean_iterations"])
            results_dict["std_simulation_iterations"].append(perf_results["std_iterations"])
    return results_dict


In [36]:
model = Model("SEIR Model")

results_dict = runPerfTests(model, seirRules, create_n_compartment_model, epiClasses)

Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 1 has finished after 50.60767701687345 days, requiring 1518 iterations and 0.1274238999467343 secs of compute time
Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 2 has finished after 10.179201893048102 days, requiring 1044 iterations and 0.08430740004405379 secs of compute time
Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 3 has finished after 36.30553637448668 days, requiring 1150 iterations and 0.09125689999200404 secs of compute time
Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 4 has finished after 22.38037988098021 days, requiring 1042 iterations and 0.08555680001154542 secs of compute time
Simulation 5 has finished after 100.

In [18]:
results_df = pd.DataFrame(results_dict)

display(results_df.iloc[list(range(36,len(results_df)))])

NameError: name 'results_dict' is not defined

In [15]:
model_sis = Model("SIS Model")

results_dict_sis = runPerfTests(model_sis, sisRules, create_n_compartment_model, epiClasses)

Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 1 has finished after 85.08936490842888 days, requiring 1990 iterations and 0.12152590020559728 secs of compute time
Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 2 has finished after 78.78283938861858 days, requiring 1990 iterations and 0.11849669995717704 secs of compute time
Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 3 has finished after 62.88909065011917 days, requiring 1990 iterations and 0.11842809990048409 secs of compute time
Finishing model simulation early.
No rules left to trigger - all rules have 0 propensity.
Ending model simulation
Simulation 4 has finished after 66.78366649282455 days, requiring 1990 iterations and 0.12338300002738833 secs of compute time
Finishing model simulation early.
No

In [17]:
results_df_sis = pd.DataFrame(results_dict_sis)

display(results_df_sis)

,model,simulation_algorithm,num_compartments,avg_simulation_time,std_simulation_time,avg_simulation_iterations,std_simulation_iterations
0,SIS Model,Direct - No Caching,1,0.122452,0.007663,1990.00,0.000000
1,SIS Model,Direct - Caching,1,0.124544,0.006483,1990.00,0.000000
2,SIS Model,First Reaction,1,0.120613,0.003791,1989.98,0.140000
3,SIS Model,Next Reaction,1,0.131601,0.003960,1990.00,0.000000
4,SIS Model,Direct - No Caching,2,0.364626,0.004582,3979.96,0.195959
5,SIS Model,Direct - Caching,2,0.273504,0.005831,3980.00,0.000000
6,SIS Model,First Reaction,2,0.270230,0.005940,3979.98,0.140000
7,SIS Model,Next Reaction,2,0.297551,0.004531,3980.00,0.000000
8,SIS Model,Direct - No Caching,3,0.757435,0.007372,5970.00,0.000000
9,SIS Model,Direct - Caching,3,0.446158,0.005148,5969.96,0.280000


In [ ]:
results_df["avg_time_per_iteration"] = results_df["avg_time_per_iteration"] / results_df["avg_simulation_iterations"]